## Bibliotecas

In [ ]:
from __future__ import annotations

import logging
from pathlib import Path
from typing import Optional

import pandas as pd
from bs4 import BeautifulSoup, Comment
from selenium import webdriver
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC 
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager


#### Config de logging


In [23]:
logger = logging.getLogger("bbr_scraper")
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)
logger.setLevel(logging.INFO)


##### Navegação e captura de HTML (com Selenium)

In [24]:
def get_rendered_html(
    url: str,
    wait_css_selector: str = "div#wrap",
    wait_timeout: int = 25,
    headless: bool = True,
    user_agent: Optional[str] = (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/127.0.0.0 Safari/537.36"
    ),
) -> str:
    """
    Renderiza a página com Chrome headless e devolve o page_source.
    """
    logger.info("Abrindo URL: %s", url)
    chrome_opts = ChromeOptions()
    if headless:
        chrome_opts.add_argument("--headless=new")
    chrome_opts.add_argument("--disable-gpu")
    chrome_opts.add_argument("--no-sandbox")
    chrome_opts.add_argument("--window-size=1920,1080")
    if user_agent:
        chrome_opts.add_argument(f"user-agent={user_agent}")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=chrome_opts
    )

    try:
        driver.get(url)
        WebDriverWait(driver, wait_timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, wait_css_selector))
        )
        html = driver.page_source
        logger.info("HTML renderizado com sucesso (%d chars).", len(html))
        return html
    finally:
        driver.quit()

#### Utilitários de parsing

In [25]:
def uncomment_tables(raw_html: str) -> BeautifulSoup:
    """
    Muitos blocos do Basketball-Reference vêm 'comentados'.
    Este passo injeta esses fragmentos de volta no DOM.
    """
    soup = BeautifulSoup(raw_html, "lxml")
    comments = soup.find_all(string=lambda t: isinstance(t, Comment))
    if comments:
        logger.info("Removendo %d bloco(s) comentado(s).", len(comments))
    for c in comments:
        soup_fragment = BeautifulSoup(c, "lxml")
        c.replace_with(soup_fragment)
    return soup


def extract_table_df(
    soup: BeautifulSoup,
    table_id: str,
    required_cols: Optional[list[str]] = None,
) -> pd.DataFrame:
    """
    Extrai uma tabela por id e valida colunas (opcional).
    Tenta fallback para 'table.stats_table' se o id não existir.
    """
    table = soup.select_one(f"table#{table_id}")
    if not table:
        logger.warning("Tabela com id '%s' não encontrada. Tentando fallback.", table_id)
        table = soup.select_one("table.stats_table")

    if not table:
        raise RuntimeError(
            f"Tabela não encontrada (id='{table_id}' e fallback 'stats_table' falharam)."
        )

    dfs = pd.read_html(str(table))
    if not dfs:
        raise RuntimeError("pd.read_html não retornou tabelas para o seletor informado.")

    df = dfs[0]
    logger.info("Tabela extraída: %s (linhas=%d, colunas=%d)", table_id, *df.shape)

    if required_cols:
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Colunas obrigatórias ausentes: {missing}")

    return df


##### Fluxo alto nível (baixa → salva → parseia)

In [26]:
def fetch_parse_bbr_table(
    url: str = "https://www.basketball-reference.com/teams/",
    out_path: Path | str = Path("teams_page.html"),
    table_id: str = "teams_active",
    wait_css_selector: str = "div#wrap",
    wait_timeout: int = 25,
    headless: bool = True,
    validate_columns: Optional[list[str]] = None,
) -> pd.DataFrame:
    """
    Baixa o HTML renderizado da página do Basketball-Reference, persiste em disco,
    descomenta tabelas e extrai a tabela desejada em DataFrame.
    """
    out_path = Path(out_path)

    # 1) Renderiza e salva HTML
    html = get_rendered_html(
        url=url, wait_css_selector=wait_css_selector, wait_timeout=wait_timeout, headless=headless
    )
    out_path.write_text(html, encoding="utf-8")
    logger.info("HTML salvo em: %s", out_path.resolve())

    # 2) Descomenta e parseia
    soup = uncomment_tables(html)
    df = extract_table_df(soup, table_id=table_id, required_cols=validate_columns)

    # 3) Higiene adicional (ex.: remover multiindex gerado por header duplo)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [" ".join(map(str, col)).strip() for col in df.columns]
        logger.info("Normalizados nomes de colunas (MultiIndex → str).")


    return df


##### Execucao

In [ ]:
URL = "https://www.basketball-reference.com/teams/"
OUT = Path("teams_page.html")
TABLE_ID = "teams_active" 
# Se quiser validar a presença de colunas específicas:
REQUIRED_COLS = None  # ex.: ["Franchise", "From", "To"]

df_teams = fetch_parse_bbr_table(
    url=URL,
    out_path=OUT,
    table_id=TABLE_ID,
    wait_css_selector="div#wrap",
    wait_timeout=25,
    headless=True,
    validate_columns=REQUIRED_COLS,
)

logger.info("Prévia:\n%s", df_teams.head().to_string(index=False))

2025-10-15 17:48:03 | INFO | Abrindo URL: https://www.basketball-reference.com/teams/
2025-10-15 17:48:09 | INFO | HTML renderizado com sucesso (294055 chars).
2025-10-15 17:48:11 | INFO | HTML salvo em: C:\Users\henri\OneDrive\Documents\Codigo\Basketanalysis\scraping\times\teams_page.html
2025-10-15 17:48:11 | INFO | Removendo 127 bloco(s) comentado(s).
C:\Users\henri\AppData\Local\Temp\ipykernel_21224\1115489816.py:35: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(table))
2025-10-15 17:48:11 | INFO | Tabela extraída: teams_active (linhas=87, colunas=13)
2025-10-15 17:48:11 | INFO | Prévia:
            Franchise  Lg    From      To Yrs    G    W    L W/L% Plyfs Div Conf Champ
        Atlanta Hawks NBA 1949-50 2025-26  77 6019 2967 3052 .493    49  12    0     1
        Atlanta Hawks NBA 1968-69 2025-26  58 4601 2269 2332 .493    36   6    

In [28]:
df

,Franchise,Lg,From,To,Yrs,G,W,L,W/L%,Plyfs,Div,Conf,Champ
0,Atlanta Hawks,NBA,1949-50,2025-26,77,6019,2967,3052,.493,49,12,0,1
1,Atlanta Hawks,NBA,1968-69,2025-26,58,4601,2269,2332,.493,36,6,0,0
2,St. Louis Hawks,NBA,1955-56,1967-68,13,1005,553,452,.550,12,6,0,1
3,Milwaukee Hawks,NBA,1951-52,1954-55,4,281,91,190,.324,0,0,0,0
4,Tri-Cities Blackhawks,NBA,1949-50,1950-51,2,132,54,78,.409,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,Washington Bullets,NBA,1974-75,1996-97,23,1886,887,999,.470,13,2,3,1
83,Capital Bullets,NBA,1973-74,1973-74,1,82,47,35,.573,1,1,0,0
84,Baltimore Bullets,NBA,1963-64,1972-73,10,813,401,412,.493,7,4,1,0
85,Chicago Zephyrs,NBA,1962-63,1962-63,1,80,25,55,.313,0,0,0,0


In [30]:
df.to_csv("../../storage/raw/team.csv", index=False)